# 🚗 Sistema OCR para Leitura de Placas Veiculares
## Segurança Condominial — Controle de Acesso

---

| Campo | Informação |
|---|---|
| **Disciplina** | MCZA018 — Processamento Digital de Imagens |
| **Instituição** | Universidade Federal do ABC — UFABC |
| **Equipe** | Caio Nunes · Edson Felipe · Nicolas Vidal |
| **Data** | Q1.2026 |
| **Arquivo** | `detectar_placa.ipynb` |
| **Chamada (Linux)** | `jupyter notebook detectar_placa.ipynb` |

---

### Descrição
Sistema de reconhecimento óptico de caracteres (OCR) aplicado à leitura de placas veiculares em imagens estáticas ou via webcam, voltado a aplicações de controle de acesso em condomínios e estacionamentos.  
O pipeline implementa os conceitos de PDI segundo **Gonzalez & Woods, 4ª ed.**:

1. **Aquisição e pré-processamento** — cinza, blur gaussiano, equalização de histograma, Blackhat morfológico  
2. **Segmentação** — binarização de Otsu + operadores morfológicos (fechamento, abertura, dilatação)  
3. **Extração de atributos** — contornos, bounding boxes, isolamento de caracteres  
4. **Reconhecimento** — Tesseract OCR com whitelist alfanumérica (PSM 7)

---
## 1. Instalação e Verificação de Dependências

In [ ]:
# Instale as dependências caso necessário
# Execute esta célula apenas uma vez no ambiente
import subprocess, sys

pkgs = ['opencv-python', 'pytesseract', 'numpy', 'matplotlib']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Dependências Python OK.')
print('\nLembrete de instalação do Tesseract:')
print('  Linux : sudo apt install tesseract-ocr')
print('  Windows: https://github.com/UB-Mannheim/tesseract/wiki')

In [ ]:
import shutil, cv2, numpy as np, pytesseract
from matplotlib import pyplot as plt

# Verificação de versões
print(f'OpenCV   : {cv2.__version__}')
print(f'NumPy    : {np.__version__}')

# Localiza o Tesseract automaticamente no PATH do OS
# Linux: /usr/bin/tesseract  |  Windows: C:\Program Files\Tesseract-OCR\tesseract.exe
pytesseract.pytesseract.tesseract_cmd = shutil.which('tesseract')

if pytesseract.pytesseract.tesseract_cmd:
    print(f'Tesseract: {pytesseract.pytesseract.tesseract_cmd}')
    print(f'Versão   : {pytesseract.get_tesseract_version()}')
else:
    print('⚠️  Tesseract NÃO encontrado. Instale antes de continuar.')

---
## 2. Configurações Globais do Sistema

Todos os parâmetros ajustáveis estão centralizados aqui.  
Modifique estes valores para calibrar o sistema a diferentes câmeras e condições de iluminação.

In [ ]:
import os

# ─── Detecção da Placa ─────────────────────────────────────────────
PROPORCAO_MIN  = 1.5     # Razão aspecto mínima (w/h) do candidato a placa
PROPORCAO_MAX  = 7.0     # Razão aspecto máxima
AREA_MIN       = 1000    # Área mínima (px²) do candidato
AREA_MAX       = 100000  # Área máxima (px²)
PAD_W_FATOR    = 0.20    # Expansão lateral do ROI (20% da largura detectada)
PAD_H_FATOR    = 0.35    # Expansão vertical do ROI (35% da altura detectada)

# ─── Morfologia ────────────────────────────────────────────────────
KERNEL_CLOSE    = (50, 20)  # Kernel de fechamento: une chars horizontalmente
ITERACOES_CLOSE = 2          # Número de iterações do fechamento

# ─── Filtro de Caracteres ──────────────────────────────────────────
ALTURA_MIN_FATOR  = 0.25  # Altura mínima do char em relação à altura do ROI
ALTURA_MAX_FATOR  = 0.60  # Altura máxima
LARGURA_MIN_FATOR = 0.02  # Largura mínima em relação à largura do ROI
LARGURA_MAX_FATOR = 0.25  # Largura máxima
PROPORCAO_CHAR_MIN = 0.1  # Razão aspecto mínima do char
PROPORCAO_CHAR_MAX = 1.2  # Razão aspecto máxima

# ─── Tesseract OCR ────────────────────────────────────────────────
# PSM 7 = trata a imagem como uma única linha de texto
# whitelist = aceita apenas A-Z e 0-9 (evita símbolos)
TESSERACT_CONFIG = r'--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'

# ─── Pasta de Imagens (modo batch) ───────────────────────────────
# Altere conforme sua estrutura de arquivos
PASTA_IMAGENS = os.path.join('img', 'Moto')  # Compatível Linux e Windows

print('Configurações carregadas:')
print(f'  Proporção placa : [{PROPORCAO_MIN}, {PROPORCAO_MAX}]')
print(f'  Área placa      : [{AREA_MIN}, {AREA_MAX}] px²')
print(f'  Kernel close    : {KERNEL_CLOSE}')
print(f'  Pasta imagens   : {PASTA_IMAGENS}')

---
## 3. Pipeline de Processamento

### 3.1 Pré-processamento — `processar_imagem()`

**Referência G&W:** Cap. 3 (filtragem espacial) + Cap. 9 (morfologia — Blackhat)

Cadeia de operações:
```
BGR → Cinza → Blur Gaussiano 5×5 → Equalização de Histograma → Blackhat (15×15)
```

O **Blackhat** realça regiões escuras em fundo claro: `blackhat(f) = fechamento(f) − f`  
Ideal para letras pretas sobre o fundo branco/amarelo das placas brasileiras.

In [ ]:
def processar_imagem(img_original):
    """
    Pré-processa a imagem para realçar os caracteres da placa.
    
    Etapas (G&W):
      - Cap. 2: conversão de espaço de cor BGR → cinza
      - Cap. 3: filtragem espacial (Blur Gaussiano) + equalização de histograma
      - Cap. 9: transformação Blackhat morfológica
    
    Args:
        img_original (np.ndarray): imagem BGR lida pelo OpenCV
    Returns:
        np.ndarray: imagem Blackhat (caracteres realçados)
    """
    # Passo 1: Converte para escala de cinza — f(x,y) ∈ [0, 255]
    img_cinza     = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

    # Passo 2: Blur Gaussiano 5×5 — reduz ruído de alta frequência
    # G(x,y) = (1/2πσ²) · exp(−(x²+y²)/2σ²)
    img_blur      = cv2.GaussianBlur(img_cinza, (5, 5), 0)

    # Passo 3: Equalização de histograma — normaliza variações de iluminação
    # s = T(r) = (L−1) · CDF(r)
    img_equalizada = cv2.equalizeHist(img_blur)

    # Passo 4: Blackhat — realça texto escuro em fundo claro
    # blackhat(f) = fechamento(f) − f
    kernel   = np.ones((15, 15), np.uint8)
    blackhat = cv2.morphologyEx(img_equalizada, cv2.MORPH_BLACKHAT, kernel)

    return blackhat

print('✅ processar_imagem() definida')

### 3.2 Detecção da Placa — `detectar_placa()`

**Referência G&W:** Cap. 10 (binarização de Otsu) + Cap. 9 (fechamento/abertura) + Cap. 11 (contornos)

Pipeline interno:
```
Blackhat → Otsu + inversão → Fechamento(50×20) → Dilatação → Abertura
         → findContours → filtro proporção/área → ROI + padding
```

In [ ]:
def detectar_placa(img_original, img_blackhat):
    """
    Detecta a região da placa na imagem.
    
    Etapas (G&W):
      - Cap. 10: binarização automática pelo método de Otsu
                 σ²_B(T) = w₀(T)·w₁(T)·[μ₀(T)−μ₁(T)]²
      - Cap. 9 : fechamento morfológico para unir caracteres em blob único
      - Cap. 11: representação por contornos e descritores de bounding box
    
    Args:
        img_original  (np.ndarray): imagem BGR original
        img_blackhat  (np.ndarray): saída de processar_imagem()
    Returns:
        placa     (np.ndarray | None): recorte BGR da placa com padding
        retangulo (tuple | None)     : (x0, y0, x1, y1) na imagem original
    """
    img_rgb = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)

    # ── Binarização de Otsu + inversão (chars = branco, fundo = preto) ──
    _, img_bin = cv2.threshold(img_blackhat, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    img_bin = cv2.bitwise_not(img_bin)

    # ── Fechamento morfológico — une todos os chars em um único blob ──
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, KERNEL_CLOSE)
    img_fechada  = cv2.morphologyEx(img_bin, cv2.MORPH_CLOSE,
                                    kernel_close, iterations=ITERACOES_CLOSE)

    # ── Dilatação + Abertura — remove ruído residual ──
    kernel_dilate = np.ones((3, 3), np.uint8)
    img_fechada   = cv2.dilate(img_fechada, kernel_dilate, iterations=1)
    kernel_open   = np.ones((3, 3), np.uint8)
    img_aberta    = cv2.morphologyEx(img_fechada, cv2.MORPH_OPEN, kernel_open)

    # ── Busca e filtragem de candidatos ──
    contornos, _ = cv2.findContours(img_aberta, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    candidatos = []
    for c in contornos:
        x, y, w, h = cv2.boundingRect(c)
        proporcao  = w / float(h)
        area       = w * h
        if PROPORCAO_MIN < proporcao < PROPORCAO_MAX and AREA_MIN < area < AREA_MAX:
            candidatos.append((x, y, w, h, area))

    # ── Seleciona o maior candidato e aplica padding ──
    placa, retangulo = None, None
    if candidatos:
        x, y, w, h, _ = max(candidatos, key=lambda b: b[4])
        pad_w = int(PAD_W_FATOR * w)
        pad_h = int(PAD_H_FATOR * h)
        x0 = max(0, x - pad_w)
        y0 = max(0, y - pad_h)
        x1 = min(img_original.shape[1], x + w + pad_w)
        y1 = min(img_original.shape[0], y + h + pad_h)
        placa     = img_original[y0:y1, x0:x1]
        retangulo = (x0, y0, x1, y1)
        cv2.rectangle(img_rgb, (x0, y0), (x1, y1), (0, 255, 0), 2)

    # ── Visualização das 4 etapas intermediárias ──
    etapas = [
        ('Original + detecção', img_rgb,     'rgb'),
        ('Binarizada (Otsu+inv)', img_bin,   'gray'),
        ('Fechamento morfológico', img_fechada, 'gray'),
        ('Abertura',             img_aberta,  'gray'),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle('Pipeline de detecção da placa', fontsize=13, fontweight='bold')
    for ax, (titulo, img, tipo) in zip(axes, etapas):
        ax.imshow(img, cmap='gray' if tipo == 'gray' else None)
        ax.set_title(titulo, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    return placa, retangulo

print('✅ detectar_placa() definida')

### 3.3 Isolamento de Caracteres — `isolar_caracteres_duas_linhas()`

**Referência G&W:** Cap. 11 (representação e descrição de regiões)

Suporta **placas de carro** (1 linha) e **placas de moto** (2 linhas) via separação pela mediana de Y dos bounding boxes.  
Cada caractere é normalizado para **40×60 px** com borda de 2 px.

In [ ]:
def isolar_caracteres_duas_linhas(placa):
    """
    Isola os caracteres individuais da placa.
    Suporta placas de carro (1 linha) e moto (2 linhas).
    
    Estratégia de separação de linhas:
      - Calcula a mediana da posição Y de todos os bounding boxes
      - Chars com y < mediana → linha superior
      - Chars com y >= mediana → linha inferior
      - Mediana é mais robusta que média a outliers (bboxes espúrios)
    
    Args:
        placa (np.ndarray): recorte BGR da placa (saída de detectar_placa)
    Returns:
        chars_sup   (list): imagens 40×60 dos chars da linha superior
        chars_inf   (list): imagens 40×60 dos chars da linha inferior
        placa_bin_inv (np.ndarray): placa binarizada invertida (debug)
        boxes       (list): bounding boxes detectados [(x,y,w,h), ...]
    """
    # ── Pré-processamento interno do ROI ──
    placa_cinza    = cv2.cvtColor(placa, cv2.COLOR_BGR2GRAY)
    placa_blur     = cv2.GaussianBlur(placa_cinza, (3, 3), 0)
    placa_eq       = cv2.equalizeHist(placa_blur)
    _, placa_bin   = cv2.threshold(placa_eq, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    placa_bin_inv  = cv2.bitwise_not(placa_bin)  # chars = branco

    # Abertura: remove ruído de borda dentro do ROI
    kernel         = np.ones((2, 2), np.uint8)
    placa_bin_inv  = cv2.morphologyEx(placa_bin_inv, cv2.MORPH_OPEN, kernel)

    # ── Busca e filtragem de bounding boxes de caracteres ──
    contornos, _   = cv2.findContours(placa_bin_inv, cv2.RETR_EXTERNAL,
                                      cv2.CHAIN_APPROX_SIMPLE)
    H, W = placa_bin_inv.shape
    boxes = []
    for c in contornos:
        x, y, w, h = cv2.boundingRect(c)
        if (ALTURA_MIN_FATOR * H <= h <= ALTURA_MAX_FATOR * H and
            LARGURA_MIN_FATOR * W <= w <= LARGURA_MAX_FATOR * W and
            PROPORCAO_CHAR_MIN <= (w / float(h)) <= PROPORCAO_CHAR_MAX):
            boxes.append((x, y, w, h))

    if not boxes:
        return [], [], placa_bin_inv, []

    # ── Separação de linhas pela mediana de Y ──
    media_y   = np.median([b[1] for b in boxes])
    linha_sup = sorted([b for b in boxes if b[1] < media_y],  key=lambda b: b[0])
    linha_inf = sorted([b for b in boxes if b[1] >= media_y], key=lambda b: b[0])

    def extrair_caracteres(linha):
        """Recorta, normaliza e inverte cada caractere para 40×60 px."""
        chars = []
        for (x, y, w, h) in linha:
            ch = placa_bin_inv[y:y+h, x:x+w]
            ch = cv2.copyMakeBorder(ch, 2, 2, 2, 2,
                                    cv2.BORDER_CONSTANT, value=0)  # borda 2px
            ch = cv2.resize(ch, (40, 60),
                            interpolation=cv2.INTER_NEAREST)        # normaliza tamanho
            ch = 255 - ch                                           # inverte: char escuro
            chars.append(ch)
        return chars

    return extrair_caracteres(linha_sup), extrair_caracteres(linha_inf), \
           placa_bin_inv, boxes

print('✅ isolar_caracteres_duas_linhas() definida')

### 3.4 OCR por Linha — `ocr_linha()`

Monta todos os caracteres de uma linha em uma única imagem horizontal (espaçamento de 5 px) e passa ao **Tesseract OCR** com:
- `PSM 7` → trata a imagem como uma única linha de texto
- `whitelist` → aceita apenas A–Z e 0–9 (evita confusão com símbolos)

In [ ]:
def ocr_linha(chars_linha):
    """
    Reconhece o texto de uma linha de caracteres via Tesseract OCR.
    
    Monta os chars em imagem horizontal → Tesseract PSM 7 → filtra alfanuméricos.
    
    Args:
        chars_linha (list): lista de imagens 40×60 (saída de isolar_caracteres)
    Returns:
        str: texto reconhecido (somente A-Z, 0-9)
    """
    if not chars_linha:
        return ''

    altura        = 60
    espaco        = 5
    largura_total = sum(ch.shape[1] for ch in chars_linha) + espaco * (len(chars_linha) - 1)

    # Cria imagem branca horizontal para receber todos os chars
    img_linha = 255 * np.ones((altura, largura_total), dtype=np.uint8)

    x_atual = 0
    for ch in chars_linha:
        h, w = ch.shape
        img_linha[0:h, x_atual:x_atual + w] = ch
        x_atual += w + espaco

    # Chamada ao Tesseract OCR
    texto = pytesseract.image_to_string(img_linha, config=TESSERACT_CONFIG)

    # Filtra apenas alfanuméricos (descarta '\n', espaços, símbolos)
    return ''.join(c for c in texto.upper() if c.isalnum())

print('✅ ocr_linha() definida')

---
## 4. Modo Batch — Processamento de Pasta de Imagens

Processa todos os arquivos `.jpg`, `.jpeg` e `.png` da pasta `PASTA_IMAGENS`.  
Para cada imagem exibe o pipeline completo de visualização e imprime o texto da placa.

In [ ]:
def processar_pasta(pasta=PASTA_IMAGENS):
    """
    Processa todas as imagens de uma pasta e exibe o pipeline completo.
    
    Args:
        pasta (str): caminho da pasta com imagens .jpg/.jpeg/.png
    """
    if not os.path.exists(pasta):
        print(f"⚠️  Pasta '{pasta}' não encontrada.")
        print("    Crie a pasta e adicione imagens de placas, ou ajuste PASTA_IMAGENS.")
        return

    arquivos = [f for f in os.listdir(pasta)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if not arquivos:
        print(f"⚠️  Nenhuma imagem encontrada em '{pasta}'.")
        return

    print(f'📂 Pasta: {pasta}  |  {len(arquivos)} imagem(ns) encontrada(s)\n')
    print('─' * 60)

    for arquivo in arquivos:
        caminho      = os.path.join(pasta, arquivo)
        print(f'\n🔍 Processando: {arquivo}')

        img_original = cv2.imread(caminho)
        if img_original is None:
            print('   ❌ Erro ao carregar imagem.')
            continue

        # ── Etapa 1: Pré-processamento ──
        img_blackhat = processar_imagem(img_original)

        # ── Etapa 2: Detecção da placa ──
        placa, _ = detectar_placa(img_original, img_blackhat)

        if placa is None:
            print('   ⚠️  Nenhuma placa detectada.')
            continue

        # ── Etapa 3: Isolamento de caracteres ──
        chars_sup, chars_inf, placa_bin_inv, boxes = \
            isolar_caracteres_duas_linhas(placa)

        # ── Visualização: ROI da placa ──
        placa_rgb = cv2.cvtColor(placa, cv2.COLOR_BGR2RGB)
        placa_vis = placa_rgb.copy()
        for (x, y, w, h) in boxes:
            cv2.rectangle(placa_vis, (x, y), (x + w, y + h), (0, 200, 0), 1)

        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        fig.suptitle(f'Placa detectada — {arquivo}', fontsize=12, fontweight='bold')
        axes[0].imshow(placa_rgb);          axes[0].set_title('Placa (ROI)');
        axes[1].imshow(placa_bin_inv, cmap='gray'); axes[1].set_title('Binarizada inv.');
        axes[2].imshow(placa_vis);          axes[2].set_title('Chars detectados');
        for ax in axes: ax.axis('off')
        plt.tight_layout()
        plt.show()

        # ── Visualização: caracteres individuais ──
        todos = chars_sup + chars_inf
        if todos:
            fig2, axes2 = plt.subplots(1, len(todos), figsize=(max(8, len(todos) * 1.2), 2.5))
            fig2.suptitle('Caracteres isolados (40×60 px)', fontsize=11)
            if len(todos) == 1:
                axes2 = [axes2]
            for ax, ch in zip(axes2, todos):
                ax.imshow(ch, cmap='gray')
                ax.axis('off')
            plt.tight_layout()
            plt.show()
        else:
            print('   ⚠️  Nenhum caractere isolado.')
            continue

        # ── Etapa 4: OCR ──
        texto_sup  = ocr_linha(chars_sup)
        texto_inf  = ocr_linha(chars_inf)
        texto_final = f'{texto_sup} {texto_inf}'.strip()

        print(f'   📋 Placa OCR: [{texto_final}]')
        print('─' * 60)


# ── Execução ──
processar_pasta(PASTA_IMAGENS)

---
## 5. Modo Câmera ao Vivo (Webcam)

Captura frames em tempo real.  
- **Tecla `c`** → captura o frame atual e executa o pipeline completo  
- **Tecla `q`** → encerra o sistema  

> ⚠️ **Atenção no Jupyter:** a janela OpenCV abre fora do notebook. Use `q` na própria janela para fechar.

In [ ]:
def modo_webcam(camera_id=0):
    """
    Captura de vídeo em tempo real via webcam.
    
    Controles:
      c → captura frame e processa o pipeline OCR
      q → encerra o sistema
    
    Args:
        camera_id (int): índice da câmera (0 = padrão)
    """
    cap = cv2.VideoCapture(camera_id)

    if not cap.isOpened():
        print(f'❌ Não foi possível abrir a câmera {camera_id}.')
        return

    print('📷 Câmera aberta. Pressione [c] para capturar | [q] para sair.')
    cv2.namedWindow('Segurança Condominial — OCR Placas', cv2.WINDOW_NORMAL)

    while True:
        ret, frame = cap.read()
        if not ret:
            print('❌ Falha ao ler frame.')
            break

        # Exibe o frame ao vivo na janela
        cv2.imshow('Segurança Condominial — OCR Placas', frame)

        tecla = cv2.waitKey(1) & 0xFF

        if tecla == ord('c'):
            print('\n📸 Frame capturado — processando...')
            img_blackhat = processar_imagem(frame)
            placa, _     = detectar_placa(frame, img_blackhat)

            if placa is not None:
                chars_sup, chars_inf, _, _ = isolar_caracteres_duas_linhas(placa)
                texto_sup  = ocr_linha(chars_sup)
                texto_inf  = ocr_linha(chars_inf)
                resultado  = f'{texto_sup} {texto_inf}'.strip()
                print(f'   📋 Placa: [{resultado}]')

                # Sobrepõe o resultado no frame
                cv2.putText(frame, resultado, (10, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
                cv2.imshow('Segurança Condominial — OCR Placas', frame)
                cv2.waitKey(2000)  # mostra resultado por 2 segundos
            else:
                print('   ⚠️  Nenhuma placa detectada neste frame.')

        elif tecla == ord('q'):
            print('\n👋 Sistema encerrado.')
            break

    cap.release()
    cv2.destroyAllWindows()


# Descomente a linha abaixo para ativar a câmera ao vivo
# modo_webcam(camera_id=0)

---
## 6. Geração de Arquivo de Vídeo (Requisito H)

Grava a saída da webcam em arquivo `.mp4` enquanto processa os frames.

In [ ]:
def gravar_video(camera_id=0, saida='saida_ocr.mp4', fps=20.0,
                 resolucao=(640, 480)):
    """
    Grava vídeo da webcam com o resultado do OCR sobreposto.
    Pressione [q] para parar a gravação.
    
    Args:
        camera_id  (int)  : índice da câmera
        saida      (str)  : nome do arquivo de saída (.mp4)
        fps        (float): frames por segundo
        resolucao  (tuple): (largura, altura) do vídeo
    """
    cap    = cv2.VideoCapture(camera_id)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(saida, fourcc, fps, resolucao)

    print(f'🎥 Gravando em "{saida}" — pressione [q] para parar.')

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, resolucao)
        out.write(frame)
        cv2.imshow('Gravando — [q] para parar', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f'✅ Vídeo salvo: {saida}')


# Descomente para gravar
# gravar_video(saida='saida_ocr.mp4')

---
## 7. Processamento de Vídeo Externo (Requisito I)

Processa um vídeo recebido de terceiros (qualquer formato suportado pelo OpenCV) e salva novo vídeo com os resultados do OCR sobrepostos.

In [ ]:
def processar_video_externo(caminho_entrada, caminho_saida='video_processado.mp4',
                            intervalo_frames=30):
    """
    Processa um vídeo externo aplicando o pipeline OCR a cada N frames.
    
    Args:
        caminho_entrada  (str): caminho do vídeo de entrada
        caminho_saida    (str): caminho do vídeo de saída
        intervalo_frames (int): aplica OCR a cada N frames (evita lentidão)
    """
    if not os.path.exists(caminho_entrada):
        print(f'❌ Vídeo não encontrado: {caminho_entrada}')
        return

    cap    = cv2.VideoCapture(caminho_entrada)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(caminho_saida, fourcc, fps, (w, h))

    print(f'📹 Processando: {caminho_entrada}')
    print(f'   Resolução: {w}×{h} | FPS: {fps:.1f} | Total frames: {total}')

    ultimo_texto = ''
    frame_idx    = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Aplica OCR a cada N frames
        if frame_idx % intervalo_frames == 0:
            img_bh = processar_imagem(frame)
            placa, _ = detectar_placa.__wrapped__(frame, img_bh) \
                       if hasattr(detectar_placa, '__wrapped__') \
                       else _detectar_placa_silencioso(frame, img_bh)
            if placa is not None:
                cs, ci, _, _ = isolar_caracteres_duas_linhas(placa)
                t = f'{ocr_linha(cs)} {ocr_linha(ci)}'.strip()
                if t:
                    ultimo_texto = t

        # Sobrepõe o último texto reconhecido em todos os frames
        if ultimo_texto:
            cv2.putText(frame, f'Placa: {ultimo_texto}', (10, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f'✅ Vídeo processado salvo: {caminho_saida}')


def _detectar_placa_silencioso(img_original, img_blackhat):
    """Versão de detectar_placa sem exibir matplotlib (para uso em vídeo)."""
    _, img_bin = cv2.threshold(img_blackhat, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    img_bin    = cv2.bitwise_not(img_bin)
    kc         = cv2.getStructuringElement(cv2.MORPH_RECT, KERNEL_CLOSE)
    img_f      = cv2.morphologyEx(img_bin, cv2.MORPH_CLOSE, kc, iterations=ITERACOES_CLOSE)
    img_f      = cv2.dilate(img_f, np.ones((3,3),np.uint8), iterations=1)
    img_a      = cv2.morphologyEx(img_f, cv2.MORPH_OPEN, np.ones((3,3),np.uint8))
    contornos, _ = cv2.findContours(img_a, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cands = [(x,y,w,h,w*h) for c in contornos
             for x,y,w,h in [cv2.boundingRect(c)]
             if PROPORCAO_MIN < w/float(h) < PROPORCAO_MAX
             and AREA_MIN < w*h < AREA_MAX]
    if not cands:
        return None, None
    x,y,w,h,_ = max(cands, key=lambda b: b[4])
    pw,ph = int(PAD_W_FATOR*w), int(PAD_H_FATOR*h)
    x0,y0 = max(0,x-pw), max(0,y-ph)
    x1,y1 = min(img_original.shape[1],x+w+pw), min(img_original.shape[0],y+h+ph)
    return img_original[y0:y1,x0:x1], (x0,y0,x1,y1)


# Descomente para processar vídeo externo
# processar_video_externo('meu_video.mp4', 'resultado.mp4')

---
## 8. Análise de Desempenho — Métrica ERL

Calcula o **Erro Relativo de Leitura (ERL)** para um conjunto de imagens com ground truth conhecido, conforme a fórmula da especificação da disciplina:

$$E_{rel} = \frac{|\text{Leitura}_{Real} - \text{Leitura}_{OCR}|}{\text{Leitura}_{Real}} \times 100\%$$

In [ ]:
def calcular_erl(texto_real, texto_ocr):
    """
    Calcula o Erro Relativo de Leitura (ERL) entre ground truth e OCR.
    Compara caractere a caractere (posição por posição).
    
    Args:
        texto_real (str): texto correto da placa (ground truth)
        texto_ocr  (str): texto lido pelo sistema OCR
    Returns:
        float: ERL em % (0% = perfeito, 100% = completamente errado)
    """
    real = texto_real.upper().replace(' ', '').replace('-', '')
    ocr  = texto_ocr.upper().replace(' ', '').replace('-', '')

    if len(real) == 0:
        return 0.0

    # Compara char a char até o comprimento do real
    erros = sum(1 for i in range(len(real))
                if i >= len(ocr) or real[i] != ocr[i])

    return (erros / len(real)) * 100.0


def avaliar_dataset(ground_truth_dict, pasta=PASTA_IMAGENS):
    """
    Avalia o sistema sobre um conjunto de imagens com ground truth.
    
    Args:
        ground_truth_dict (dict): {nome_arquivo: 'TEXTO_REAL', ...}
        pasta (str): pasta das imagens
    """
    resultados = []
    print(f'{'Arquivo':<25} | {'Ground Truth':<12} | {'OCR':<12} | {'ERL %':>6}')
    print('─' * 64)

    for arquivo, gt in ground_truth_dict.items():
        caminho = os.path.join(pasta, arquivo)
        img     = cv2.imread(caminho)
        if img is None:
            print(f'{arquivo:<25} | {gt:<12} | {"ERRO LEITURA":<12} | {"─":>6}')
            continue

        bh          = processar_imagem(img)
        placa, _    = _detectar_placa_silencioso(img, bh)
        texto_ocr   = ''

        if placa is not None:
            cs, ci, _, _ = isolar_caracteres_duas_linhas(placa)
            texto_ocr    = f'{ocr_linha(cs)}{ocr_linha(ci)}'

        erl = calcular_erl(gt, texto_ocr)
        resultados.append(erl)
        status = '✅' if erl == 0 else ('⚠️' if erl < 50 else '❌')
        print(f'{status} {arquivo:<23} | {gt:<12} | {texto_ocr or "ND":<12} | {erl:>5.1f}%')

    if resultados:
        media = sum(resultados) / len(resultados)
        acertos = sum(1 for e in resultados if e == 0)
        print('─' * 64)
        print(f'   ERL médio: {media:.1f}%  |  '
              f'Leituras corretas: {acertos}/{len(resultados)} '
              f'({acertos/len(resultados)*100:.0f}%)')


# Exemplo de uso — substitua pelos seus arquivos e placas reais:
ground_truth_exemplo = {
    # 'nome_arquivo.jpg': 'TEXTOREAL',
    # 'placa_abc1d23.jpg': 'ABC1D23',
    # 'placa_moto.jpg'   : 'XYZ1234',
}

if ground_truth_exemplo:
    avaliar_dataset(ground_truth_exemplo)
else:
    print('ℹ️  Preencha o dicionário ground_truth_exemplo com seus arquivos e placas.')
    print('   Exemplo: {"minha_placa.jpg": "ABC1D23"}')

---
## 9. Teste Rápido com Imagem Única

Aponte para uma imagem específica para testar o pipeline passo a passo.

In [ ]:
# ── Altere o caminho abaixo para testar uma imagem específica ──
CAMINHO_TESTE = os.path.join('img', 'Carro', 'exemplo.jpg')

if os.path.exists(CAMINHO_TESTE):
    print(f'Testando: {CAMINHO_TESTE}\n')

    img = cv2.imread(CAMINHO_TESTE)

    # Etapa 1 — Pré-processamento
    bh = processar_imagem(img)

    # Etapa 2 — Detecção (exibe pipeline)
    placa, rect = detectar_placa(img, bh)

    if placa is not None:
        print(f'Placa detectada em: {rect}')

        # Etapa 3 — Caracteres
        chars_sup, chars_inf, placa_bin, boxes = isolar_caracteres_duas_linhas(placa)
        print(f'Chars linha sup: {len(chars_sup)}  |  Chars linha inf: {len(chars_inf)}')

        # Etapa 4 — OCR
        t_sup = ocr_linha(chars_sup)
        t_inf = ocr_linha(chars_inf)
        print(f'\n🏁 Resultado OCR: [{t_sup} {t_inf}]'.strip())
    else:
        print('⚠️  Placa não detectada nesta imagem.')
else:
    print(f'⚠️  Arquivo não encontrado: {CAMINHO_TESTE}')
    print('   Ajuste CAMINHO_TESTE para um arquivo existente.')

---
## 10. Resumo do Pipeline e Referências

| Etapa | Função | Operação | G&W 4ª ed. |
|---|---|---|---|
| Aquisição | `cv2.imread` / `VideoCapture` | Leitura BGR | Cap. 2 |
| Conversão de cor | `cv2.cvtColor` | BGR → Cinza | Cap. 2 |
| Filtragem | `cv2.GaussianBlur` | Passa-baixa 5×5 | Cap. 3 |
| Histograma | `cv2.equalizeHist` | Equalização | Cap. 3 |
| Blackhat | `cv2.morphologyEx(MORPH_BLACKHAT)` | Fechamento − f | Cap. 9 |
| Segmentação | `cv2.threshold(THRESH_OTSU)` | Binarização automática | Cap. 10 |
| Fechamento | `cv2.morphologyEx(MORPH_CLOSE)` | Une caracteres | Cap. 9 |
| Abertura | `cv2.morphologyEx(MORPH_OPEN)` | Remove ruído | Cap. 9 |
| Contornos | `cv2.findContours` | Representação | Cap. 11 |
| Bounding box | `cv2.boundingRect` | Descritor de forma | Cap. 11 |
| OCR | `pytesseract.image_to_string` | Reconhecimento | Cap. 12 |

---

### Referências

- GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. 4. ed. Pearson, 2018.
- BRADSKI, G.; KAEHLER, A. **Learning OpenCV 3**. O'Reilly Media, 2016.
- SMITH, R. An Overview of the Tesseract OCR Engine. *ICDAR*, 2007.
- OTSU, N. A Threshold Selection Method from Gray-Level Histograms. *IEEE Trans. SMC*, v. 9, n. 1, 1979.